In [105]:
import re                               # for document class
import xml.etree.ElementTree as ET      # for parsing xml
import os
import pandas as pd
# text processor
#import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
# end text processor
# encoplot engine
import subprocess
from sentence_transformers import SentenceTransformer, util
import torch
# end encoplot engine
import shutil                             # copy dataset
# for dbscan
from sklearn.cluster import DBSCAN
import numpy as np
# prefilter with tfidf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import math
import time
from collections import defaultdict
# for ngram hashing
import hashlib

In [2]:
from google.colab import drive

# mount the drive
drive.mount('/content/drive', force_remount=True)

# define the paths for the dataset
BASE_PATH = "/content/drive/MyDrive/PAN11/external-detection-corpus"

SOURCE_PATH = os.path.join(BASE_PATH, "source-document")
SUSPICIOUS_PATH = os.path.join(BASE_PATH, "suspicious-document")

print(f"Lookin for data in: {BASE_PATH}")

Mounted at /content/drive
Lookin for data in: /content/drive/MyDrive/PAN11/external-detection-corpus


In [103]:
# define the paths for the encoplot code
ENCOPLOT_PATH = "/content/drive/MyDrive/utils/encoplot.c"
EXECUTABLE_PATH = "./encoplot_engine"

SBERT_THRESHOLD = 0.65
EPS = 10000
MIN_SAMPLE = 3
CONFIDENCE_THRESHOLD = 0.75
TFIDF_THRESHOLD = 0.05

MAX_ANCHORS = 1000
FRAGMENT_WINDOW = 300
BATCH_SIZE = 1024
MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'     #paraphrase-multilingual-mpnet-base-v2  # paraphrase-multilingual-MiniLM-L12-v2

In [4]:
!gcc -O3 {ENCOPLOT_PATH} -o {EXECUTABLE_PATH}

In [5]:
%%writefile mass_encoplot.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <omp.h>

#define halfword_t unsigned long long
typedef struct { halfword_t lo, hi; } word_t;

#define eqword(x,y) (x.lo==y.lo && x.hi==y.hi)
#define ltword(x,y) (x.hi<y.hi || (x.hi==y.hi && x.lo<y.lo))
#define readat(x,y,z) {x.lo=*(halfword_t *)(y+z); x.hi=*(halfword_t *)(y+z+8);}

typedef struct {
    unsigned char *buf;
    int *ix;
    int len;
} processed_doc;

void simpler_rsort(unsigned char *x, int l, int DEPTH, int *retval) {
    int NN = l - DEPTH + 1;
    if (NN <= 0) return;
    int *ox = (int*)malloc(NN * sizeof(int));
    int counters[256], startpos[256];
    for (int i = 0; i < NN; i++) retval[i] = i;
    for (int j = 0; j < DEPTH; j++) {
        memset(counters, 0, sizeof(counters));
        for (int i = 0; i < NN; i++) counters[x[j + retval[i]]]++;
        int sp = 0;
        for (int k = 0; k < 256; k++) { startpos[k] = sp; sp += counters[k]; }
        for (int i = 0; i < NN; i++) {
            unsigned char c = x[j + retval[i]];
            ox[startpos[c]++] = retval[i];
        }
        memcpy(retval, ox, NN * sizeof(int));
    }
    free(ox);
}

processed_doc process_file(const char *path) {
    processed_doc doc = {NULL, NULL, 0};
    FILE *f = fopen(path, "rb");
    if (!f) return doc;
    fseek(f, 0, SEEK_END);
    doc.len = ftell(f);
    rewind(f);
    doc.buf = (unsigned char *)malloc(doc.len + 20);
    fread(doc.buf, 1, doc.len, f);
    fclose(f);
    doc.ix = (int *)malloc(doc.len * sizeof(int));
    simpler_rsort(doc.buf, doc.len, 16, doc.ix);
    return doc;
}

int main(int argc, char **argv) {
    if (argc < 3) {
        fprintf(stderr, "Using: %s suspicious.txt source1.txt source2.txt ...\n", argv[0]);
        return 1;
    }

    processed_doc susp = process_file(argv[1]);
    if (!susp.buf) return 1;

    int depth = 16;
    int susp_limit = susp.len - (depth - 1);

    #pragma omp parallel for schedule(dynamic)
    for (int i = 2; i < argc; i++) {
        processed_doc src = process_file(argv[i]);
        if (!src.buf) continue;

        int src_limit = src.len - (depth - 1);
        int i1 = 0, i2 = 0;

        while (i1 < susp_limit && i2 < src_limit) {
            word_t s1; readat(s1, susp.buf, susp.ix[i1]);
            word_t s2; readat(s2, src.buf, src.ix[i2]);

            if (eqword(s1, s2)) {
                #pragma omp critical
                printf("%d %d %d\n", i, susp.ix[i1], src.ix[i2]);
                i1++; i2++;
            } else if (ltword(s1, s2)) { i1++; } else { i2++; }
        }
        free(src.buf); free(src.ix);
    }

    free(susp.buf); free(susp.ix);
    return 0;
}

Writing mass_encoplot.c


In [6]:
!gcc -O3 -fopenmp mass_encoplot.c -o mass_encoplot

mass_encoplot.c: In function ‘process_file’:
mass_encoplot.c:47:5: warning: ignoring return value of ‘fread’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   47 |     fread(doc.buf, 1, doc.len, f);
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [7]:
class Document:
  def __init__(self, doc_id, is_source=True):
    self.numeric_id = int(doc_id)
    self.is_source = is_source

    formatted_id = f"{self.numeric_id:05d}"
    prefix = "source-document" if is_source else "suspicious-document"

    part_number = ((self.numeric_id - 1) // 500) + 1
    part_folder = f"part{part_number}"
    base_folder = SOURCE_PATH if is_source else SUSPICIOUS_PATH

    self.doc_name = f"{prefix}{formatted_id}.txt"
    self.xml_name = f"{prefix}{formatted_id}.xml"

    self.file_path = os.path.join(base_folder, part_folder, self.doc_name)
    self.xml_path = os.path.join(base_folder, part_folder, self.xml_name)

    with open(self.file_path, "r", encoding='utf-8', errors='ignore') as f:
      self.text = f.read()

    self.metadata = self._parse_xml()
    self.language = self.metadata.get('lang', 'english')
    self.segments = []

  def _parse_xml(self):
    meta = {}
    self.plagiarism_features = []
    try:
        tree = ET.parse(self.xml_path)
        root = tree.getroot()

        for feature in root.findall('feature'):
            if feature.get('name') == 'about':
                meta['lang'] = feature.get('lang', 'en')

            if feature.get('name') == 'md5Hash':
                meta['md5'] = feature.get('value')

            if feature.get('name') == 'plagiarism':
                self.plagiarism_features.append(PlagiarismFeature(
                    this_offset=int(feature.get('this_offset')),
                    this_length=int(feature.get('this_length')),
                    source_reference=feature.get('source_reference'),
                    source_offset=int(feature.get('source_offset')),
                    source_length=int(feature.get('source_length')),
                    obfuscation=feature.get('obfuscation', 'none')
                ))
    except Exception as e:
        print(f"Error parsing XML: {e}")
    return meta

In [8]:
class PlagiarismFeature:
    def __init__(self, this_offset, this_length, source_reference,
                 source_offset, source_length, obfuscation):
        self.this_offset = this_offset
        self.this_length = this_length
        self.source_reference = source_reference
        self.source_offset = source_offset
        self.source_length = source_length
        self.obfuscation = obfuscation

    def get_source_id(self):
        return int(self.source_reference
                       .replace("source-document", "")
                       .replace(".txt", ""))

    def __repr__(self):
        return (f"PlagiarismFeature("
                f"offset={self.this_offset}, "
                f"length={self.this_length}, "
                f"source={self.source_reference}, "
                f"obfuscation={self.obfuscation})")

In [9]:
class PredictedSegment:
    def __init__(self, susp_id, src_id, susp_off, susp_len, src_off, src_len, score):
        self.susp_id = susp_id
        self.src_id = src_id
        self.susp_off = susp_off
        self.susp_len = susp_len
        self.src_off = src_off
        self.src_len = src_len
        self.score = score

    def __repr__(self):
        return f"<Match Susp:{self.susp_id} Src:{self.src_id} Score:{self.score:.2f}>"

In [10]:
class EncoplotEngine:
    def __init__(self, executable_path="./mass_encoplot", max_anchors=MAX_ANCHORS):
        self.executable = executable_path
        self.max_anchors = max_anchors

    def run_mass_scan(self, susp_path, source_paths):
        cmd = [self.executable, susp_path] + source_paths
        result = subprocess.run(cmd, capture_output=True, text=True)
        anchors_by_source = {path: [] for path in source_paths}

        if result.returncode != 0:
            print(f"  [!] Encoplot execution error: {result.stderr}")
            return anchors_by_source

        for line in result.stdout.strip().split('\n'):
            if not line: continue
            parts = list(map(int, line.split()))
            if len(parts) == 3:
                s_idx, susp_off, src_off = parts
                anchors_by_source[source_paths[s_idx - 2]].append([susp_off, src_off])
        return anchors_by_source

    def apply_bucketing(self, anchors, doc_len):
        if len(anchors) <= self.max_anchors: return anchors
        bucket_size = max(1, doc_len // self.max_anchors)
        buckets = {}
        for p_susp, p_src in anchors:
            bucket_idx = p_susp // bucket_size
            if bucket_idx not in buckets: buckets[bucket_idx] = (p_susp, p_src)
        return list(buckets.values())



In [91]:
def cluster_results_dbscan(hits, eps=EPS, min_samples=MIN_SAMPLE, confidence_threshold=CONFIDENCE_THRESHOLD):
    if not hits:
        return []

    points = np.array([[h.susp_off, 0] for h in hits])

    db = DBSCAN(eps=eps, min_samples=min_samples).fit(points)
    labels = db.labels_

    clustered_segments = []
    unique_labels = set(labels)

    for label in unique_labels:
        if label == -1:
            continue

        cluster_hits = [hits[i] for i in range(len(hits)) if labels[i] == label]
        if not cluster_hits:
            continue

        ref_hit = cluster_hits[0]

        susp_min = min(h.susp_off for h in cluster_hits)
        susp_max = max(h.susp_off + h.susp_len for h in cluster_hits)

        src_min = min(h.src_off for h in cluster_hits)
        src_max = max(h.src_off + h.src_len for h in cluster_hits)

        avg_score = sum(h.score for h in cluster_hits) / len(cluster_hits)

        clustered_segments.append(PredictedSegment(
            susp_id=ref_hit.susp_id,
            src_id=ref_hit.src_id,
            susp_off=susp_min,
            susp_len=susp_max - susp_min,
            src_off=src_min,
            src_len=src_max - src_min,
            score=avg_score
        ))

    return clustered_segments

In [12]:
class SemanticAnalyzer:
  def __init__(self, model_name=MODEL_NAME):
    self.device = "cuda" if torch.cuda.is_available() else "cpu"
    self.model = SentenceTransformer(model_name).to(self.device)

In [98]:
def get_safe_fragment(text, offset, win=FRAGMENT_WINDOW):
        if not text:
            return "", 0, 0

        text_len = len(text)
        start = max(0, min(offset, text_len - 1))

        while start > 0:
            if text[start-1] in [' ', '\n', '\t']:
                break
            start -= 1

        end = min(text_len, start + win)

        while end < len(text) and text[end] not in [' ', '\n', '.', '!', '?']:
            end += 1

        fragment = text[start:end]
        return fragment, start, end - start

In [81]:
class ValidationMetrics:
    def __init__(self):
        pass

    def compute_granularity(self, gt_fragments, detected_fragments):
        if not gt_fragments:
            return 1.0
        total_gran = 0
        for gt_start, gt_end in gt_fragments:
            overlapping = 0
            for det_start, det_end in detected_fragments:
                if det_start < gt_end and det_end > gt_start:
                    overlapping += 1
            total_gran += max(1, overlapping)
        return total_gran / len(gt_fragments)

    def calculate_metrics(self, ground_truth_features, predicted_segments):
        metrics = {
            "recall": 0.0,
            "precision": 0.0,
            "f1": 0.0,
            "granularity": 1.0,
            "plagdet": 0.0
        }

        if not ground_truth_features:
            metrics["recall"] = 1.0
            return metrics

        # recall
        total_recall = 0
        for gt in ground_truth_features:
            covered_len = 0
            for pred in predicted_segments:
                intersect_start = max(gt.this_offset, pred.susp_off)
                intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)
                if intersect_end > intersect_start:
                    covered_len += (intersect_end - intersect_start)
            total_recall += (covered_len / gt.this_length)
        metrics["recall"] = total_recall / len(ground_truth_features)

        # precision
        if predicted_segments:
            total_precision = 0
            for pred in predicted_segments:
                covered_len = 0
                for gt in ground_truth_features:
                    intersect_start = max(gt.this_offset, pred.susp_off)
                    intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)
                    if intersect_end > intersect_start:
                        covered_len += (intersect_end - intersect_start)
                total_precision += (covered_len / pred.susp_len)
            metrics["precision"] = total_precision / len(predicted_segments)

        # granularity
        gt_tuples = [(f.this_offset, f.this_offset + f.this_length) for f in ground_truth_features]
        det_tuples = [(d.susp_off, d.susp_off + d.susp_len) for d in predicted_segments]
        metrics["granularity"] = self.compute_granularity(gt_tuples, det_tuples)

        # f1 and plagDet
        if metrics["precision"] + metrics["recall"] > 0:
            metrics["f1"] = 2 * (metrics["precision"] * metrics["recall"]) / (metrics["precision"] + metrics["recall"])

        metrics["plagdet"] = metrics["f1"] / math.log2(1 + metrics["granularity"]) if metrics["granularity"] > 0 else 0

        return {k: round(v, 4) for k, v in metrics.items()}


    def calculate_global_score(self, all_results_list):
      relevant_results = [res for res in all_results_list if not (res['recall'] == 1.0 and res['precision'] == 0.0)]

      if not relevant_results:
        print("\n" + "!"*60)
        print("  NO RELEVANT DATA TO CALCULATE GLOBAL SCORE")
        print("  (All documents were original and correctly identified as such)")
        print("!"*60)
        return None

      n = len(relevant_results)
      avg_metrics = {k: sum(res[k] for res in relevant_results) / n for k in relevant_results[0].keys()}

      print("\n" + "="*60)
      print(f"GLOBAL SCORE (average for {n} relevant documents)")
      print("-" * 60)
      print(f"  Precision    : {avg_metrics['precision']:.4f}")
      print(f"  Recall       : {avg_metrics['recall']:.4f}")
      print(f"  F1-Score     : {avg_metrics['f1']:.4f}")
      print(f"  Granularity  : {avg_metrics['granularity']:.4f}")
      print(f"  PlagDet      : {avg_metrics['plagdet']:.4f}")
      print("="*60 + "\n")

      return avg_metrics

    def analyze_by_obfuscation(self, suspicious_docs, detections):
        import math

        obf_data = {}

        for doc in suspicious_docs:
            susp_id = doc.numeric_id
            my_dets = detections.get(susp_id, [])

            for gt in doc.plagiarism_features:
                obf_type = gt.obfuscation if gt.obfuscation else "unknown"

                if obf_type not in obf_data:
                    obf_data[obf_type] = {
                        'gt_count': 0, 'det_count': 0,
                        'gt_fragments': [], 'det_fragments': [],
                        'gt_chars': set(), 'det_chars': set()
                    }

                obf_data[obf_type]['gt_count'] += 1
                gt_start, gt_end = gt.this_offset, gt.this_offset + gt.this_length
                obf_data[obf_type]['gt_fragments'].append((gt_start, gt_end))

                for c in range(gt_start, gt_end):
                    obf_data[obf_type]['gt_chars'].add((susp_id, c))

            for det in my_dets:
                det_start, det_end = det.susp_off, det.susp_off + det.susp_len
                assigned_obf = None
                max_overlap = 0

                for gt in doc.plagiarism_features:
                    overlap = max(0, min(gt.this_offset + gt.this_length, det_end) - max(gt.this_offset, det_start))
                    if overlap > max_overlap:
                        max_overlap = overlap
                        assigned_obf = gt.obfuscation if gt.obfuscation else "unknown"

                if assigned_obf is None:
                    assigned_obf = "false_positive_only"

                if assigned_obf not in obf_data:
                    obf_data[assigned_obf] = {
                        'gt_count': 0, 'det_count': 0,
                        'gt_fragments': [], 'det_fragments': [],
                        'gt_chars': set(), 'det_chars': set()
                    }

                obf_data[assigned_obf]['det_count'] += 1
                obf_data[assigned_obf]['det_fragments'].append((det_start, det_end))
                for c in range(det_start, det_end):
                    obf_data[assigned_obf]['det_chars'].add((susp_id, c))

        print("\n" + "="*85)
        print("DEEP PERFORMANCE ANALYSIS BY PLAGIARISM OBFUSCATION TYPE")
        print("-" * 85)
        print(f"{'Obfuscation Type':<18} | {'GT/Det':<8} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10} | {'Gran':<6} | {'PlagDet':<8}")
        print("-" * 85)

        for obf_type, data in obf_data.items():
            if obf_type == "false_positive_only":
                continue

            gt_chars = data['gt_chars']
            det_chars = data['det_chars']

            tp_chars = len(gt_chars & det_chars)

            precision = tp_chars / len(det_chars) if len(det_chars) > 0 else 0.0
            recall = tp_chars / len(gt_chars) if len(gt_chars) > 0 else 0.0

            f1 = 0.0
            if precision + recall > 0:
                f1 = 2 * (precision * recall) / (precision + recall)

            gran = self.compute_granularity(data['gt_fragments'], data['det_fragments'])

            plagdet = f1 / math.log2(1 + gran) if gran > 0 else 0.0

            print(f"{obf_type:<18} | {data['gt_count']}/{data['det_count']:<6} | {precision:.4f}    | {recall:.4f} | {f1:.4f}   | {gran:.2f} | {plagdet:.4f}")

        print("="*85 + "\n")


In [15]:
def get_path(doc_id, is_source=True):
    prefix = "source" if is_source else "suspicious"
    part_num = ((int(doc_id) - 1) // 500) + 1
    file_name = f"{prefix}-document{int(doc_id):05d}.txt"
    return os.path.join(BASE_PATH, f"{prefix}-document", f"part{part_num}", file_name)

In [82]:
def extract_sample(n_suspicious):
    suspicious_docs = []
    source_ids_needed = set()

    for i in range(1, n_suspicious + 1):
        try:
            doc = Document(i, is_source=False)
            suspicious_docs.append(doc)

            for pf in doc.plagiarism_features:
                src_id = int(pf.source_reference
                               .replace("source-document", "")
                               .replace(".txt", ""))
                source_ids_needed.add(src_id)

        except Exception as e:
            print(f"[!] Suspicious {i:05d} error: {e}")
            continue

    print(f"Loaded suspicious docs : {len(suspicious_docs)}")
    print(f"Unique sources : {len(source_ids_needed)}")

    return suspicious_docs, sorted(source_ids_needed)

In [83]:
encoplot = EncoplotEngine()
analyzer = SemanticAnalyzer()
validator = ValidationMetrics()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
def prefilter_with_tfidf(texts_susp, texts_src, all_meta, tfidf_threshold=TFIDF_THRESHOLD):
    print(f"Prefilltering with tf-idf on {len(texts_susp)} pairs")

    if not texts_susp:
        return [], [], []

    all_texts = texts_susp + texts_src
    vectorizer = TfidfVectorizer(max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    n = len(texts_susp)
    susp_matrix = tfidf_matrix[:n]
    src_matrix  = tfidf_matrix[n:]

    filtered_susp, filtered_src, filtered_meta = [], [], []

    BATCH = 10000
    for i in range(0, n, BATCH):
        batch_susp = susp_matrix[i:i+BATCH]
        batch_src  = src_matrix[i:i+BATCH]

        scores = np.array(batch_susp.multiply(batch_src).sum(axis=1)).flatten()

        for j, score in enumerate(scores):
            if score >= tfidf_threshold:
                filtered_susp.append(texts_susp[i+j])
                filtered_src.append(texts_src[i+j])
                filtered_meta.append(all_meta[i+j])

    print(f"after tf-idf: {len(filtered_susp)} remaining pairs (crossed out: {n - len(filtered_susp)})")
    return filtered_susp, filtered_src, filtered_meta

In [84]:
def debug_plagiarism(doc_susp, detections, all_meta_pre_tfidf, all_meta_post_tfidf, scores_sbert=None, threshold=SBERT_THRESHOLD):
    print(f"DETAILED ANALYSIS: {doc_susp.doc_name} ---")

    my_detections = detections.get(doc_susp.numeric_id, [])

    if doc_susp.plagiarism_features:
        print(f"Ground Truth properties: {len(doc_susp.plagiarism_features)} plagiarized fragments.")
        for pf in doc_susp.plagiarism_features:
            gt_start = pf.this_offset
            gt_end = pf.this_offset + pf.this_length
            src_target = pf.get_source_id()

            enc_hits = [m for m in all_meta_pre_tfidf if m['susp_id'] == doc_susp.numeric_id
                        and m['src_id'] == src_target
                        and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]
            tfidf_hits = [m for m in all_meta_post_tfidf if m['susp_id'] == doc_susp.numeric_id
                          and m['src_id'] == src_target
                          and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]

            best_score = 0.0
            if scores_sbert is not None:
                relevant_indices = [i for i, m in enumerate(all_meta_post_tfidf)
                                   if m['susp_id'] == doc_susp.numeric_id
                                   and m['src_id'] == src_target
                                   and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]
                if relevant_indices:
                    best_score = max([scores_sbert[i].item() for i in relevant_indices])

            final_det = [d for d in my_detections if d.src_id == src_target
                         and (d.susp_off >= gt_start - 500 and d.susp_off <= gt_end + 500)]

            status = " DETECTED" if final_det else " MISSED"
            print(f"\nGT Fragment [{gt_start}:{gt_end}] (Source {src_target:05d}) -> {status}")
            print(f"   - Raw anchors (Encoplot): {len(enc_hits)}")
            print(f"   - After TF-IDF filter:    {len(tfidf_hits)}")
            if len(tfidf_hits) > 0:
                print(f"   - Best SBERT score:       {best_score:.4f}")

            if not final_det:
                if not enc_hits:
                    print("   [CAUSE]: Lexical Failure - Encoplot found no anchors.")
                elif not tfidf_hits:
                    print("   [CAUSE]: Statistical Failure - TF-IDF removed the fragment.")
                else:
                    print(f"   [CAUSE]: Semantic Failure - Score {best_score:.4f} < {threshold} or DBSCAN removed it.")
    else:
        print("Original document (according to Ground Truth).")

    for det in my_detections:
        is_real = False
        for pf in doc_susp.plagiarism_features:
            if not (det.susp_off + det.susp_len < pf.this_offset or pf.this_offset + pf.this_length < det.susp_off):
                is_real = True
                break

        if not is_real:
            print(f"\n FALSE POSITIVE DETECTED (Wrongly reported as plagiarism):")
            print(f"   - Position: [{det.susp_off}:{det.susp_off + det.susp_len}] | Source: {det.src_id:05d}")
            print(f"   - SBERT score of the error: {det.score:.4f}")


In [85]:
class NgramRadarEngine:
    def __init__(self, n_gram_size=4, max_anchors=MAX_ANCHORS):
        self.n = n_gram_size
        self.max_anchors = max_anchors

    def _get_ngrams_with_offsets(self, text):
        words_iter = re.finditer(r'\b\w+\b', text.lower())
        words = [(m.group(0), m.start()) for m in words_iter]

        ngram_map = defaultdict(list)

        if len(words) < self.n:
            return ngram_map

        for i in range(len(words) - self.n + 1):
            ngram_phrase = " ".join([w[0] for w in words[i:i+self.n]])
            ngram_hash = hashlib.md5(ngram_phrase.encode('utf-8')).hexdigest()[:12]
            start_offset = words[i][1]
            ngram_map[ngram_hash].append(start_offset)

        return ngram_map

    def run_mass_scan(self, susp_path, source_paths):
        anchors_by_source = {path: [] for path in source_paths}

        try:
            with open(susp_path, "r", encoding='utf-8', errors='ignore') as f:
                susp_text = f.read()
        except Exception:
            return anchors_by_source

        susp_ngrams = self._get_ngrams_with_offsets(susp_text)

        if not susp_ngrams:
            return anchors_by_source

        for src_path in source_paths:
            try:
                with open(src_path, "r", encoding='utf-8', errors='ignore') as f:
                    src_text = f.read()
            except Exception:
                continue

            # Extract n-grams from the source
            src_ngrams = self._get_ngrams_with_offsets(src_text)

            # Find common hashes (intersections)
            common_hashes = set(susp_ngrams.keys()).intersection(set(src_ngrams.keys()))

            # Generate the [susp_off, src_off] pairs
            for h in common_hashes:
                susp_occ = susp_ngrams[h]
                src_occ = src_ngrams[h]

                # THE DYNAMIC CAP SOLUTION
                # If the phrase is relatively unique (<= 15 times in both docs)
                # Keep ALL occurrences. This catches verbatim text deep in books.
                if len(susp_occ) <= 15 and len(src_occ) <= 15:
                    for s_off in susp_occ:
                        for src_off in src_occ:
                            anchors_by_source[src_path].append([s_off, src_off])

                # If it's super common grammar noise (> 15 times)
                # Cap it heavily to prevent combinatorial explosions
                else:
                    for s_off in susp_occ[:3]:
                        for src_off in src_occ[:3]:
                            anchors_by_source[src_path].append([s_off, src_off])

            # Sort the anchors by suspicious offset to match Encoplot's output style
            anchors_by_source[src_path].sort(key=lambda x: x[0])

        return anchors_by_source

    def apply_bucketing(self, anchors, doc_len):
        """
        Identical to your current bucketing logic to ensure SBERT doesn't run OOM.
        """
        if len(anchors) <= self.max_anchors:
            return anchors

        bucket_size = max(1, doc_len // self.max_anchors)
        buckets = {}
        for p_susp, p_src in anchors:
            bucket_idx = p_susp // bucket_size
            if bucket_idx not in buckets:
                buckets[bucket_idx] = (p_susp, p_src)
        return list(buckets.values())

In [99]:
def run_pipeline(n_suspicious, analyzer=None):
    if analyzer is None:
        if 'analyzer' in globals():
            analyzer = globals()['analyzer']
            print("[INFO] Using existing analyzer found in memory.")
        else:
            print("[INFO] Initializing new SemanticAnalyzer...")
            analyzer = SemanticAnalyzer()

    print("-" * 80)
    print(f"[STEP 1] Extracting sample: {n_suspicious} suspicious files")
    print("-" * 80)
    suspicious_docs, source_ids = extract_sample(n_suspicious)
    print(f"Necessary sources: {len(source_ids)}\n")

    print("[STEP 2] Loading source docs...")
    source_docs = []
    for s_id in source_ids:
        try:
            source_docs.append(Document(s_id, is_source=True))
        except Exception as e:
            print(f"  [!] Error loading source {s_id}: {e}")
    print(f"Sources loaded: {len(source_docs)}\n")

    print("[STEP 3] N-gram Hashing - Native Python Lexical Radar")
    t3 = time.time()
    # You can tune n_gram_size (3, 4, 5). 5 is usually a good balance for text.
    engine = NgramRadarEngine(n_gram_size=4, max_anchors=MAX_ANCHORS)

    all_texts_susp = []
    all_texts_src = []
    all_meta_pre_tfidf = []

    for doc_susp in suspicious_docs:
        src_paths = [d.file_path for d in source_docs]
        raw_results = engine.run_mass_scan(doc_susp.file_path, src_paths)

        for s_path, anchors in raw_results.items():
            if not anchors: continue

            raw_anchor_count = len(anchors)
            anchors = engine.apply_bucketing(anchors, len(doc_susp.text))
            s_id = int(re.search(r'document(\d+)', s_path).group(1))

            for p_susp, p_src in anchors:
                f_susp, s_off, s_len = get_safe_fragment(doc_susp.text, p_susp)
                doc_src = next((d for d in source_docs if d.numeric_id == s_id), None)
                if not doc_src: continue
                f_src, src_off, src_len = get_safe_fragment(doc_src.text, p_src)

                if not f_susp or not f_src: continue

                t_susp = re.sub(r'\s+', ' ', f_susp).strip().lower()
                t_src  = re.sub(r'\s+', ' ', f_src).strip().lower()

                if len(t_susp) > 30:
                    all_texts_susp.append(t_susp)
                    all_texts_src.append(t_src)
                    all_meta_pre_tfidf.append({
                        'susp_id': doc_susp.numeric_id, 'src_id': s_id,
                        's_off': s_off, 's_len': s_len,
                        'src_off': src_off, 'src_len': src_len,
                        'raw_anchors': raw_anchor_count
                    })

    t3_elapsed = time.time() - t3
    print(f"Candidate fragments from Encoplot: {len(all_texts_susp)} | Time: {t3_elapsed/60:.1f}min\n")

    print(f"[STEP 4] Smart Prefiltering")
    bypass_susp, bypass_src, bypass_meta = [], [], []
    to_filter_susp, to_filter_src, to_filter_meta = [], [], []

    DENSITY_BYPASS_THRESHOLD = 6

    for i in range(len(all_texts_susp)):
        if all_meta_pre_tfidf[i]['raw_anchors'] >= DENSITY_BYPASS_THRESHOLD:
            bypass_susp.append(all_texts_susp[i])
            bypass_src.append(all_texts_src[i])
            bypass_meta.append(all_meta_pre_tfidf[i])
        else:
            to_filter_susp.append(all_texts_susp[i])
            to_filter_src.append(all_texts_src[i])
            to_filter_meta.append(all_meta_pre_tfidf[i])

    print(f"[STEP 4] Prefiltering strategy:")
    print(f"  - {len(bypass_susp)} dense pairs bypass TF-IDF automatically.")
    print(f"  - {len(to_filter_susp)} sparse pairs sent to TF-IDF checking.")

    if len(to_filter_susp) > 0:
        filtered_susp, filtered_src, filtered_meta = prefilter_with_tfidf(
            to_filter_susp, to_filter_src, to_filter_meta, tfidf_threshold=TFIDF_THRESHOLD
        )
    else:
        filtered_susp, filtered_src, filtered_meta = [], [], []

    all_texts_susp_post_tfidf = bypass_susp + filtered_susp
    all_texts_src_post_tfidf  = bypass_src + filtered_src
    all_meta_post_tfidf       = bypass_meta + filtered_meta

    print(f"Total pairs remaining for SBERT: {len(all_meta_post_tfidf)} "
          f"(Dropped by TF-IDF: {len(to_filter_susp) - len(filtered_susp)})\n")

    print(f"[STEP 5] SBERT batch encoding for {len(all_texts_susp_post_tfidf)} pairs...")
    t5 = time.time()

    emb_susp = analyzer.model.encode(all_texts_susp_post_tfidf, convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=True, batch_size=BATCH_SIZE)
    emb_src  = analyzer.model.encode(all_texts_src_post_tfidf,  convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=True, batch_size=BATCH_SIZE)

    print("[INFO] Computing cosine similarities in chunks")
    scores = np.zeros(len(all_texts_susp_post_tfidf), dtype=np.float32)
    chunk_size = 50000
    for i in range(0, len(all_texts_susp_post_tfidf), chunk_size):
        j = min(i + chunk_size, len(all_texts_susp_post_tfidf))
        dot = np.sum(emb_susp[i:j] * emb_src[i:j], axis=1)
        n_s = np.linalg.norm(emb_susp[i:j], axis=1)
        n_src = np.linalg.norm(emb_src[i:j], axis=1)
        scores[i:j] = dot / (n_s * n_src + 1e-8)

    del emb_susp
    del emb_src
    torch.cuda.empty_cache()

    print(f"Encoding done in {time.time() - t5:.1f}s\n")

    print("[STEP 6] Filtering and clustering with DBSCAN")
    hits_per_pair = defaultdict(list)
    for i, score in enumerate(scores):
        if score >= SBERT_THRESHOLD:
            m = all_meta_post_tfidf[i]
            hits_per_pair[(m['susp_id'], m['src_id'])].append(
                PredictedSegment(m['susp_id'], m['src_id'], m['s_off'], m['s_len'], m['src_off'], m['src_len'], score)
            )

    detections = defaultdict(list)
    for (susp_id, src_id), hits in hits_per_pair.items():
        clustered = cluster_results_dbscan(hits)
        detections[susp_id].extend(clustered)
    print(f"Pairs with detected plagiarism: {len(hits_per_pair)}\n")

    print("[STEP 7] Generating XML output...")
    os.makedirs("/content/output", exist_ok=True)
    for doc_susp in suspicious_docs:
        root = ET.Element("document", reference=doc_susp.doc_name)
        for det in detections.get(doc_susp.numeric_id, []):
            ET.SubElement(root, "feature",
                name="detected-plagiarism",
                this_offset=str(det.susp_off), this_length=str(det.susp_len),
                source_reference=f"source-document{det.src_id:05d}.txt",
                source_offset=str(det.src_off), source_length=str(det.src_len)
            )
        ET.ElementTree(root).write(f"/content/output/{doc_susp.doc_name.replace('.txt', '.xml')}", encoding='utf-8', xml_declaration=True)

    all_results = []
    print("[STEP 8] Detected vs ground truth...")
    print("-"*60)

    for doc_susp in suspicious_docs:
        m = validator.calculate_metrics(doc_susp.plagiarism_features, detections.get(doc_susp.numeric_id, []))
        all_results.append(m)
        if len(doc_susp.plagiarism_features) > 0 or len(detections.get(doc_susp.numeric_id, [])) > 0:
            print(f"  Suspicious {doc_susp.numeric_id:05d} | GT: {len(doc_susp.plagiarism_features)} | "
                  f"Det: {len(detections.get(doc_susp.numeric_id, []))} | P: {m['precision']:.2f} R: {m['recall']:.2f}")

    validator.calculate_global_score(all_results)

    print("\n[STEP 9] Running obfuscation analysis...")
    validator.analyze_by_obfuscation(suspicious_docs, detections)

    print("\n" + "="*40)
    print("DETAILED TRACING OF GROUND TRUTH SEGMENTS")
    print("="*40)
    for doc_susp in suspicious_docs:
        debug_plagiarism(doc_susp, detections, all_meta_pre_tfidf, all_meta_post_tfidf, scores, SBERT_THRESHOLD)

    return suspicious_docs, source_docs, detections, scores, all_meta_post_tfidf

In [101]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections, cache_scores, cache_meta = run_pipeline(
    n_suspicious=5,
    analyzer=analyzer
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 5 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 5
Unique sources : 1
Necessary sources: 1

[STEP 2] Loading source docs...
Sources loaded: 1

[STEP 3] N-gram Hashing - Native Python Lexical Radar
Candidate fragments from Encoplot: 140 | Time: 0.0min

[STEP 4] Smart Prefiltering
[STEP 4] Prefiltering strategy:
  - 138 dense pairs bypass TF-IDF automatically.
  - 2 sparse pairs sent to TF-IDF checking.
Prefilltering with tf-idf on 2 pairs
after tf-idf: 2 remaining pairs (crossed out: 0)
Total pairs remaining for SBERT: 140 (Dropped by TF-IDF: 0)

[STEP 5] SBERT batch encoding for 140 pairs...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[INFO] Computing cosine similarities in chunks
Encoding done in 1.6s

[STEP 6] Filtering and clustering with DBSCAN
Pairs with detected plagiarism: 1

[STEP 7] Generating XML output...
[STEP 8] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00005 | GT: 1 | Det: 1 | P: 0.92 R: 0.99

GLOBAL SCORE (average for 1 relevant documents)
------------------------------------------------------------
  Precision    : 0.9159
  Recall       : 0.9929
  F1-Score     : 0.9529
  Granularity  : 1.0000
  PlagDet      : 0.9529


[STEP 9] Running obfuscation analysis...

DEEP PERFORMANCE ANALYSIS BY PLAGIARISM OBFUSCATION TYPE
-------------------------------------------------------------------------------------
Obfuscation Type   | GT/Det   | Precision  | Recall     | F1-Score   | Gran   | PlagDet 
-------------------------------------------------------------------------------------
low                | 1/1      | 0.9159    | 0.9929 | 0.9529   | 1.00 | 

In [102]:
import time
from collections import defaultdict

if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections, cache_scores, cache_meta = run_pipeline(
    n_suspicious=10,
    analyzer=analyzer
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 10 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 10
Unique sources : 3
Necessary sources: 3

[STEP 2] Loading source docs...
Sources loaded: 3

[STEP 3] N-gram Hashing - Native Python Lexical Radar
Candidate fragments from Encoplot: 5745 | Time: 0.3min

[STEP 4] Smart Prefiltering
[STEP 4] Prefiltering strategy:
  - 5737 dense pairs bypass TF-IDF automatically.
  - 8 sparse pairs sent to TF-IDF checking.
Prefilltering with tf-idf on 8 pairs
after tf-idf: 8 remaining pairs (crossed out: 0)
Total pairs remaining for SBERT: 5745 (Dropped by TF-IDF: 0)

[STEP 5] SBERT batch encoding for 5745 pairs...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

[INFO] Computing cosine similarities in chunks
Encoding done in 76.2s

[STEP 6] Filtering and clustering with DBSCAN
Pairs with detected plagiarism: 10

[STEP 7] Generating XML output...
[STEP 8] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00004 | GT: 0 | Det: 1 | P: 0.00 R: 1.00
  Suspicious 00005 | GT: 1 | Det: 1 | P: 0.92 R: 0.99
  Suspicious 00006 | GT: 0 | Det: 1 | P: 0.00 R: 1.00
  Suspicious 00007 | GT: 6 | Det: 1 | P: 0.94 R: 1.00
  Suspicious 00010 | GT: 6 | Det: 1 | P: 0.78 R: 1.00

GLOBAL SCORE (average for 3 relevant documents)
------------------------------------------------------------
  Precision    : 0.8808
  Recall       : 0.9967
  F1-Score     : 0.9337
  Granularity  : 1.0000
  PlagDet      : 0.9337


[STEP 9] Running obfuscation analysis...

DEEP PERFORMANCE ANALYSIS BY PLAGIARISM OBFUSCATION TYPE
-------------------------------------------------------------------------------------
Obfuscation Type   | GT/Det 

In [95]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections, cache_scores, cache_meta = run_pipeline(
    n_suspicious=30,
    analyzer=analyzer
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 30 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 30
Unique sources : 21
Necessary sources: 21

[STEP 2] Loading source docs...
Sources loaded: 21

[STEP 3] N-gram Hashing - Native Python Lexical Radar
Candidate fragments from Encoplot: 49840 | Time: 2.1min

[STEP 4] Smart Prefiltering
[STEP 4] Prefiltering strategy:
  - 49471 dense pairs bypass TF-IDF automatically.
  - 369 sparse pairs sent to TF-IDF checking.
Prefilltering with tf-idf on 369 pairs
after tf-idf: 359 remaining pairs (crossed out: 10)
Total pairs remaining for SBERT: 49830 (Dropped by TF-IDF: 10)

[STEP 5] SBERT batch encoding for 49830 pairs...


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

[INFO] Computing cosine similarities in chunks
Encoding done in 739.0s

[STEP 6] Filtering and clustering with DBSCAN
Pairs with detected plagiarism: 74

[STEP 7] Generating XML output...
[STEP 8] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00004 | GT: 0 | Det: 3 | P: 0.00 R: 1.00
  Suspicious 00005 | GT: 1 | Det: 1 | P: 0.88 R: 0.99
  Suspicious 00006 | GT: 0 | Det: 1 | P: 0.00 R: 1.00
  Suspicious 00007 | GT: 6 | Det: 1 | P: 0.94 R: 1.00
  Suspicious 00010 | GT: 6 | Det: 1 | P: 0.78 R: 1.00
  Suspicious 00011 | GT: 12 | Det: 17 | P: 0.57 R: 1.01
  Suspicious 00012 | GT: 14 | Det: 3 | P: 0.98 R: 0.82
  Suspicious 00014 | GT: 3 | Det: 2 | P: 0.31 R: 0.89
  Suspicious 00015 | GT: 11 | Det: 2 | P: 0.36 R: 0.71
  Suspicious 00016 | GT: 1 | Det: 1 | P: 0.95 R: 0.97
  Suspicious 00020 | GT: 3 | Det: 1 | P: 0.76 R: 1.00
  Suspicious 00024 | GT: 3 | Det: 6 | P: 0.29 R: 0.64
  Suspicious 00025 | GT: 1 | Det: 1 | P: 1.00 R: 0.20
  Suspic

In [ ]:
import nbformat

with open('/content/drive/MyDrive/Colab Notebooks/licenta.ipynb', 'r') as f:
    nb = nbformat.read(f, as_version=4)

if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

for cell in nb.cells:
    if 'metadata' in cell:
        if 'executionInfo' in cell.metadata:
            del cell.metadata['executionInfo']

with open('/content/drive/MyDrive/Colab Notebooks/licenta_clean.ipynb', 'w') as f:
    nbformat.write(nb, f)

print("Done!")

Done!
